**Sample users and foods from the filtered lists and pair them according to their nutrition tags**

**User groups by major status:**
- Obesity
- Hypertension
- Diabetes
- Opioid Misuse

**Primary nutrition tags for both users and foods:**
- carb
- sugar
- sodium
- cholesterol
- calorie
- protein

**How to define easy/medium/hard questions?**
- Easy(10/17): single tag matching/contradictive
- Medium(5/17): multiple tags all matching/contradictive
- Hard(2/17): multiple tags containing match&contradict


---
Load the user and food data

In [25]:
import pandas as pd


users_w_tags = pd.read_csv('../processed_data/user_tagging.csv')
food_list = pd.read_csv('../task_2_foods_filtering_qa_creation/processed_data/reduced_mixed_dishes_v4.csv')

categories of the user tags

In [2]:
primary_nutrition_tags = [
    'low_carb ', 
    'low_sugar ',
    'low_sodium ', 
    'low_cholesterol ',
    'low_calorie ', 'high_calorie ',
    'low_protein ', 'high_protein ', 
]

micro_nutrition_tags = [
    'high_fiber ',
    'low_saturated_fat ', 
    'high_calcium ', 
    'low_phosphorus ', 
    'high_potassium ', 
    'high_iron ',
    'high_folate_acid ',
    'high_vitamin_c ', 
    'high_vitamin_d', 
    'high_vitamin_b12 ', 
]

user_nutrition_tags = primary_nutrition_tags + micro_nutrition_tags

auxiliary_tags = [
    # 'Weight loss/Low calorie diet ',
    # 'Low fat/Low cholesterol diet ',
    # 'Low salt/Low sodium diet ',
    # 'Sugar free/Low sugar diet ',
    # 'Diabetic diet ',
    # 'Weight gain/Muscle building diet ',
    # 'Low carbohydrate diet ',
    # 'High protein diet ',
    # 'Renal/Kidney diet ',
    'opioid_misuse ',
    'hypertension ',
    'diabetes ',
    'obesity ',
]

user_tags = user_nutrition_tags + auxiliary_tags

Filter out users with no primary nutrition tags

In [3]:
user_list = []
for index, row in users_w_tags.iterrows():
    user_item = row.to_dict()
    user_nutrition_dict = {}
    user_nutrition_dict['user_id'] = user_item['SEQN   ']

    for nutrition_tag in primary_nutrition_tags:
        if user_item[nutrition_tag] == 1:
            if 'low' in nutrition_tag:
                user_nutrition_dict[nutrition_tag[4:-1]] = -1
            else:
                user_nutrition_dict[nutrition_tag[5:-1]] = 1
    # print(user_nutrition_dict)

    # we only pick users with at least one tag
    if len(list(user_nutrition_dict.keys())) > 1:
        user_list.append(user_nutrition_dict)

num_candidate_user = len(user_list)
print(user_list[:5])
print(num_candidate_user)

[{'user_id': 100000, 'calorie': -1}, {'user_id': 100001, 'sodium': -1, 'calorie': -1}, {'user_id': 100004, 'sodium': -1}, {'user_id': 100009, 'calorie': -1}, {'user_id': 100012, 'sugar': -1, 'calorie': -1}]
36313


Categories of the food tags

In [23]:
primary_nutrition_tags = [
    'low_carb', 'high_carb',
    'low_sugar', 'high_sugar',
    'low_sodium', 'high_sodium',
    'low_calorie', 'high_calorie',
    'low_protein', 'high_protein',
    'low_cholesterol', 'high_cholesterol',
]

micro_nutrition_tags = [
    'low_fiber', 'high_fiber',
    'low_saturated_fat', 'high_saturated_fat',
    'low_calcium', 'high_calcium',
    'low_phosphorus', 'high_phosphorus',
    'low_potassium', 'high_potassium',
    'low_iron', 'high_iron',
    'low_folic_acid', 'high_folic_acid',
    'low_vitamin_c', 'high_vitamin_c',
    'low_vitamin_d', 'high_vitamin_d',
    'low_vitamin_b12', 'high_vitamin_b12',
]

We intend to sample (10 + 5 + 2) users for each food, and we do this by using the 3 `while` loops in the code below

In [28]:
import random

easy_count = 10
medium_count = 5
hard_count = 2
pair_list = []

for index, row in food_list.iterrows():
    food_item = row.to_dict()
    food_nutrition_dict = {}

    # convert high/low_xxx tags to uniform xxx tags
    for nutrition_tag in primary_nutrition_tags:
        if food_item[nutrition_tag] == 1:
            if 'low' in nutrition_tag:
                food_nutrition_dict[nutrition_tag[4:]] = -1
            else:
                food_nutrition_dict[nutrition_tag[5:]] = 1
    # print(food_item['food_id'], food_nutrition_dict)

    pair_dict = {}
    pair_dict['food_id'] = food_item['food_id']
    pair_dict['food_tag'] = food_nutrition_dict
    pair_dict['easy'] = []
    pair_dict['medium'] = []
    pair_dict['hard'] = []

    nutrition_list = list(food_nutrition_dict.keys())

    # generate easy pairs (only 1 tag in common)
    # print('Generating easy pairs...')
    loop_seed = pair_dict['food_id']
    while(True):
        loop_seed += 1
        # shuffle the nutrition tag list of this food
        random.seed(loop_seed)
        random.shuffle(nutrition_list)

        # pick the first element as the nutrition tag aligning with the user's tag
        selected_nutrition = [nutrition_list[0]]
        avoid_nutrition = nutrition_list[1:]

        # select users only involve selected nutrition tags
        for user in random.sample(user_list, num_candidate_user):
            continue_flag = False
            for nutrition in avoid_nutrition:
                if nutrition in user.keys():
                    continue_flag = True
                    break
            if continue_flag:
                continue
            
            for nutrition in selected_nutrition:
                if nutrition not in user.keys():
                    continue_flag = True
                    break
            if continue_flag:
                continue

            pair_dict['easy'].append(user)
            break
        
        if len(pair_dict['easy']) == easy_count or loop_seed > 50 + pair_dict['food_id']:
            break
    
    # no medium or hard sample if the food has only one tag
    if len(nutrition_list) == 1:
        continue

    # generate medium pairs (the food's tag of random number[2, max] in common, same high/low)
    # print('Generating medium pairs...')
    loop_seed = pair_dict['food_id']
    while(True):
        loop_seed += 1
        random.seed(loop_seed)
        random.shuffle(nutrition_list)

        num_tag_in_common = random.randint(2, len(list(food_nutrition_dict.keys())))
        selected_nutrition = nutrition_list[:num_tag_in_common]
        avoid_nutrition = nutrition_list[num_tag_in_common:]
        if not isinstance(avoid_nutrition, list):
            avoid_nutrition = [avoid_nutrition]

        for user in random.sample(user_list, num_candidate_user):
            continue_flag = False
            for nutrition in avoid_nutrition:
                if nutrition in user.keys():
                    continue_flag = True
                    break
            if continue_flag:
                continue

            for nutrition in selected_nutrition:
                if nutrition not in user.keys():
                    continue_flag = True
                    break
            if continue_flag:
                continue

            j=0
            for nutrition in selected_nutrition:
                if food_nutrition_dict[nutrition] == user[nutrition]:
                    j += 1
                elif food_nutrition_dict[nutrition] == -user[nutrition]:
                    j -= 1
            if abs(j) == len(selected_nutrition):
                pair_dict['medium'].append(user)
                break
        
        if len(pair_dict['medium']) == medium_count or loop_seed > 50 + pair_dict['food_id']:
            break
    
    # generate hard pairs (the food's tag of random number[2, max] in common)
    # print('Generating hard pairs...')
    loop_seed = pair_dict['food_id']
    while(True):
        loop_seed += 1
        random.seed(loop_seed)
        random.shuffle(nutrition_list)

        num_tag_in_common = random.randint(2, len(list(food_nutrition_dict.keys())))
        selected_nutrition = nutrition_list[:num_tag_in_common]
        avoid_nutrition = nutrition_list[num_tag_in_common:]
        if not isinstance(avoid_nutrition, list):
            avoid_nutrition = [avoid_nutrition]

        for user in random.sample(user_list, num_candidate_user):
            continue_flag = False
            for nutrition in avoid_nutrition:
                if nutrition in user.keys():
                    continue_flag = True
                    break
            if continue_flag:
                continue

            for nutrition in selected_nutrition:
                if nutrition not in user.keys():
                    continue_flag = True
                    break
            if continue_flag:
                continue

            j=0
            for nutrition in selected_nutrition:
                if food_nutrition_dict[nutrition] == user[nutrition]:
                    j += 1
                elif food_nutrition_dict[nutrition] == -user[nutrition]:
                    j -= 1
            if abs(j) != len(selected_nutrition):
                pair_dict['hard'].append(user)
                break
        
        if len(pair_dict['hard']) == hard_count or loop_seed > 50 + pair_dict['food_id']:
            break
    
    
    pair_list.append(pair_dict)
print(pair_list[:2])


[{'food_id': 58137300, 'food_tag': {'carb': -1, 'sugar': -1, 'sodium': 1, 'protein': 1, 'cholesterol': 1}, 'easy': [{'user_id': 59453, 'cholesterol': -1}, {'user_id': 70010, 'sugar': -1}, {'user_id': 118110, 'sugar': -1}, {'user_id': 74388, 'protein': -1}, {'user_id': 113442, 'cholesterol': -1, 'calorie': -1}, {'user_id': 90746, 'carb': -1}, {'user_id': 102429, 'sugar': -1, 'calorie': -1}, {'user_id': 66729, 'sodium': -1}, {'user_id': 78337, 'cholesterol': -1}, {'user_id': 27452, 'protein': -1}], 'medium': [{'user_id': 111717, 'cholesterol': -1, 'protein': -1}, {'user_id': 97606, 'sodium': -1, 'protein': -1}, {'user_id': 101548, 'carb': -1, 'sugar': -1, 'calorie': -1}, {'user_id': 22231, 'sodium': -1, 'cholesterol': -1, 'protein': -1}, {'user_id': 122136, 'sodium': -1, 'cholesterol': -1}], 'hard': [{'user_id': 53110, 'carb': -1, 'sugar': -1, 'sodium': -1, 'protein': -1}, {'user_id': 65642, 'carb': -1, 'sugar': -1, 'cholesterol': -1, 'calorie': -1, 'protein': -1}]}, {'food_id': 58150530

Raw pairs statistics (contain duplicates)

In [29]:
easy_sum = 0
medium_sum = 0
hard_sum = 0

for food_user_dict in pair_list:
    easy_sum += len(food_user_dict['easy'])
    medium_sum += len(food_user_dict['medium'])
    hard_sum += len(food_user_dict['hard'])

print(easy_sum, medium_sum, hard_sum)

6620 3251 1310


Save the pairs data as json file

In [8]:
import json

with open('processed_data/user_food_pair.json', 'w') as f:
    json.dump(pair_list, f)